# FedUCA Phase 5: Federated Temperature Scaling (CPU only)

**Add data:** `ham10000-vit-embeddings-orig` + `feduca-phase2-results`

**Accelerator: NONE (CPU)** — no GPU needed, runs in ~30 minutes.

Computes ECE before/after calibration and melanoma triage sensitivity.

In [ ]:
import os, json, glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_phase5"
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE      = torch.device("cpu")   # CPU only
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
NUM_CLASSES = 7
MELANOMA    = 4   # index of melanoma class

# Auto-find embeddings
hits = glob.glob("/kaggle/input/**/embeddings_train.npz", recursive=True)
if not hits:
    raise FileNotFoundError("Add ham10000-vit-embeddings-orig dataset first.")
EMBED_DIR = os.path.dirname(hits[0])

# Auto-find Phase 2 weights
wt_hits = glob.glob("/kaggle/input/**/weights_bottleneck_r4_a1000_s0.pt",
                    recursive=True)
if not wt_hits:
    raise FileNotFoundError("Add feduca-phase2-results dataset first.")
P2_DIR = os.path.dirname(wt_hits[0])

print(f"Device    : CPU")
print(f"Embeddings: {EMBED_DIR}")
print(f"P2 weights: {P2_DIR}")

In [ ]:
# Load embeddings
def load_split(name):
    d = np.load(f"{EMBED_DIR}/embeddings_{name}.npz", allow_pickle=True)
    return (torch.tensor(d['embeddings'], dtype=torch.float32),
            torch.tensor(d['labels'],     dtype=torch.long))

emb_train, lbl_train = load_split("train")
emb_val,   lbl_val   = load_split("val")
emb_test,  lbl_test  = load_split("test")

print(f"Train: {emb_train.shape} | Val: {emb_val.shape} | Test: {emb_test.shape}")

counts  = torch.bincount(lbl_train, minlength=NUM_CLASSES).float()
weights = 1.0 / counts
weights = (weights / weights.mean())

In [ ]:
# Model
class BottleneckMLP(nn.Module):
    def __init__(self, ratio=4):
        super().__init__()
        b = 768 // ratio
        self.net = nn.Sequential(
            nn.Linear(768, 4*b), nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, b),   nn.GELU(), nn.LayerNorm(b),   nn.Dropout(0.2),
            nn.Linear(b, 4*b),   nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, 7))
    def forward(self, x): return self.net(x)

def dirichlet_partition(labels, K, alpha, seed):
    rng = np.random.default_rng(seed)
    idx = [[] for _ in range(K)]
    for c in range(int(labels.max()) + 1):
        ci = np.where(labels.numpy() == c)[0]
        rng.shuffle(ci)
        props  = rng.dirichlet(np.full(K, alpha))
        splits = (np.cumsum(props) * len(ci)).astype(int)[:-1]
        for k, chunk in enumerate(np.split(ci, splits)):
            idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

@torch.no_grad()
def get_logits(head, embs, bs=512):
    head.eval()
    out = []
    for x, in DataLoader(TensorDataset(embs), batch_size=bs):
        out.append(head(x))
    return torch.cat(out)   # (N, 7)

print("Model ready.")

In [ ]:
# Calibration metrics
# FIXED: find_T now minimises NLL (not ECE) — matches manuscript Section 3.4
# FIXED: melanoma_sensitivity uses mel-class probability, not max-class probability

def nll(logits, labels, T=1.0):
    """Negative log-likelihood as function of temperature T."""
    scaled = logits / T
    log_probs = torch.log_softmax(scaled, dim=1)
    return -log_probs[torch.arange(len(labels)), labels].mean().item()

def ece(logits, labels, T=1.0, n_bins=15):
    """Top-label ECE with 15 equal-width bins; bins <5 samples excluded."""
    probs        = torch.softmax(logits / T, dim=1)
    conf, pred   = probs.max(dim=1)
    correct      = (pred == labels)
    edges        = torch.linspace(0, 1, n_bins + 1)
    ece_val      = 0.0
    for i in range(n_bins):
        mask = (conf >= edges[i]) & (conf < edges[i+1])
        if mask.sum() >= 5:   # exclude bins with <5 samples
            acc       = correct[mask].float().mean()
            conf_mean = conf[mask].mean()
            ece_val  += float(mask.sum() / len(labels)) * abs(float(acc) - float(conf_mean))
    return float(ece_val)

def brier_score(logits, labels, T=1.0):
    probs = torch.softmax(logits / T, dim=1)
    onehot = torch.zeros_like(probs)
    onehot.scatter_(1, labels.unsqueeze(1), 1.0)
    return float(((probs - onehot)**2).mean())

def reliability_data(logits, labels, T=1.0, n_bins=15):
    probs        = torch.softmax(logits / T, dim=1)
    conf, pred   = probs.max(dim=1)
    correct      = (pred == labels)
    edges        = torch.linspace(0, 1, n_bins + 1)
    centres, accs, confs, counts = [], [], [], []
    for i in range(n_bins):
        mask = (conf >= edges[i]) & (conf < edges[i+1])
        centres.append(float((edges[i] + edges[i+1]) / 2))
        if mask.sum() > 0:
            accs.append(float(correct[mask].float().mean()))
            confs.append(float(conf[mask].mean()))
            counts.append(int(mask.sum()))
        else:
            accs.append(0.0); confs.append(0.0); counts.append(0)
    return centres, accs, confs, counts

def find_T(logits, labels):
    """
    FIX: Minimise NLL (not ECE) with respect to temperature T.
    NLL is convex in s=1/T; we verify unimodality on a dense grid
    before golden-section search, as described in manuscript Section 3.4.
    """
    from scipy.optimize import minimize_scalar
    # Dense grid check for unimodality
    T_grid = torch.linspace(0.5, 5.0, 200)
    nll_grid = [nll(logits, labels, float(t)) for t in T_grid]
    min_idx = int(torch.tensor(nll_grid).argmin())
    # Confirm not at boundary
    assert 0 < min_idx < 199, f"WARNING: NLL minimum at boundary T={T_grid[min_idx]:.2f}"
    # Golden-section search for precision
    result = minimize_scalar(
        lambda T: nll(logits, labels, float(T)),
        bounds=(0.5, 5.0), method="bounded",
        options={"xatol": 1e-6})
    return float(result.x)

def melanoma_sensitivity(logits, labels, T=1.0, threshold=0.70):
    """
    FIX: Use melanoma-class probability (not max-class probability).
    Sensitivity = fraction of true melanoma cases where P(mel) >= threshold.
    """
    probs    = torch.softmax(logits / T, dim=1)
    mel_prob = probs[:, MELANOMA]           # melanoma-class probability
    mel_mask = (labels == MELANOMA)
    if mel_mask.sum() == 0:
        return float("nan")
    return float((mel_prob[mel_mask] >= threshold).float().mean())

def specificity(logits, labels, T=1.0, threshold=0.70):
    """
    FIX: Use melanoma-class probability (not max-class probability).
    Specificity = fraction of non-melanoma cases where P(mel) < threshold.
    """
    probs    = torch.softmax(logits / T, dim=1)
    mel_prob = probs[:, MELANOMA]           # melanoma-class probability
    non_mel  = (labels != MELANOMA)
    if non_mel.sum() == 0:
        return float("nan")
    return float((mel_prob[non_mel] < threshold).float().mean())

print("Calibration functions defined (NLL-based find_T, mel-class sensitivity).")


In [ ]:
# Main experiment: three temperature strategies across seeds and alpha
ALPHAS = [1000, 1, 0.1]
SEEDS  = [0, 1, 2, 3, 4]
K      = 4
all_cal = []

print("="*60)
print("PHASE 5: FEDERATED TEMPERATURE SCALING")
print("="*60)

for alpha in ALPHAS:
    print(f"\n=== alpha={alpha} ===")
    for seed in SEEDS:
        wpath = os.path.join(P2_DIR,
                             f"weights_bottleneck_r4_a{alpha}_s{seed}.pt")
        if not os.path.exists(wpath):
            print(f"  WARNING: {wpath} not found — skipping")
            continue

        head = BottleneckMLP(ratio=4)
        head.load_state_dict(torch.load(wpath, map_location="cpu"))
        head.eval()

        val_logits  = get_logits(head, emb_val)
        test_logits = get_logits(head, emb_test)

        # Per-client val partition (same seed as training)
        part        = dirichlet_partition(lbl_val, K, alpha, seed)
        c_logits    = [val_logits[idx]  for idx in part]
        c_labels    = [lbl_val[idx]     for idx in part]
        c_sizes     = [len(idx)          for idx in part]

        # Strategy A — no scaling (T=1)
        ece_a_val   = ece(val_logits,  lbl_val,  T=1.0)
        ece_a_test  = ece(test_logits, lbl_test, T=1.0)
        sens_a      = melanoma_sensitivity(test_logits, lbl_test, T=1.0)
        spec_a      = specificity(test_logits, lbl_test, T=1.0)

        # Strategy B — global T (single T minimising ECE on full val set)
        T_global    = find_T(val_logits, lbl_val)
        ece_b_val   = ece(val_logits,  lbl_val,  T=T_global)
        ece_b_test  = ece(test_logits, lbl_test, T=T_global)
        sens_b      = melanoma_sensitivity(test_logits, lbl_test, T=T_global)
        spec_b      = specificity(test_logits, lbl_test, T=T_global)

        # Strategy C — federated T (weighted avg of per-client T*)
        client_Ts   = []
        for k in range(K):
            if c_sizes[k] > 5:
                Tk = find_T(c_logits[k], c_labels[k])
            else:
                Tk = 1.0
            client_Ts.append(Tk)
        total       = sum(c_sizes)
        T_fed       = float(sum((c_sizes[k]/total)*client_Ts[k] for k in range(K)))
        ece_c_val   = ece(val_logits,  lbl_val,  T=T_fed)
        ece_c_test  = ece(test_logits, lbl_test, T=T_fed)
        sens_c      = melanoma_sensitivity(test_logits, lbl_test, T=T_fed)
        spec_c      = specificity(test_logits, lbl_test, T=T_fed)

        # Reliability diagram data (for plotting)
        rel_a = reliability_data(test_logits, lbl_test, T=1.0)
        rel_c = reliability_data(test_logits, lbl_test, T=T_fed)

        entry = {
            "alpha": alpha, "seed": seed,
            "no_scaling":   {"T": 1.0,       "ECE_val": ece_a_val,
                             "ECE_test": ece_a_test,
                             "mel_sens": sens_a, "specificity": spec_a},
            "global_T":     {"T": T_global,  "ECE_val": ece_b_val,
                             "ECE_test": ece_b_test,
                             "mel_sens": sens_b, "specificity": spec_b},
            "federated_T":  {"T": T_fed,     "ECE_val": ece_c_val,
                             "ECE_test": ece_c_test,
                             "mel_sens": sens_c, "specificity": spec_c,
                             "client_Ts": client_Ts},
            "reliability_no_scaling":  rel_a,
            "reliability_federated_T": rel_c,
        }
        all_cal.append(entry)

        with open(os.path.join(RESULTS_DIR,
                               f"cal_a{alpha}_s{seed}.json"), "w") as f:
            json.dump(entry, f, indent=2)

        print(f"  seed={seed} | T_fed={T_fed:.3f} | "
              f"ECE: {ece_a_test:.4f} -> {ece_c_test:.4f} | "
              f"mel_sens: {sens_a:.3f} -> {sens_c:.3f}")

with open(os.path.join(RESULTS_DIR, "all_calibration.json"), "w") as f:
    json.dump(all_cal, f, indent=2)
print("\nAll calibration results saved.")

In [ ]:
# Summary tables
print("\n" + "="*60)
print("ECE SUMMARY (mean over 5 seeds)")
print("="*60)
print(f"{'alpha':<8} {'No scaling':>12} {'Global T':>12} {'Federated T':>12} {'Improvement':>13}")
print("-"*57)

for alpha in ALPHAS:
    rows = [r for r in all_cal if r['alpha']==alpha]
    if not rows: continue
    ns = np.mean([r['no_scaling' ]['ECE_test'] for r in rows])
    gt = np.mean([r['global_T'   ]['ECE_test'] for r in rows])
    ft = np.mean([r['federated_T']['ECE_test'] for r in rows])
    imp = (ns - ft) / ns * 100
    print(f"{alpha:<8} {ns:>12.4f} {gt:>12.4f} {ft:>12.4f} {imp:>12.1f}%")

print("\n" + "="*60)
print("MELANOMA SENSITIVITY @ 70% confidence (mean over 5 seeds)")
print("="*60)
print(f"{'alpha':<8} {'No scaling':>12} {'Global T':>12} {'Federated T':>12}")
print("-"*44)
for alpha in ALPHAS:
    rows = [r for r in all_cal if r['alpha']==alpha]
    if not rows: continue
    ns = np.mean([r['no_scaling' ]['mel_sens'] for r in rows])
    gt = np.mean([r['global_T'   ]['mel_sens'] for r in rows])
    ft = np.mean([r['federated_T']['mel_sens'] for r in rows])
    print(f"{alpha:<8} {ns:>12.3f} {gt:>12.3f} {ft:>12.3f}")

print("\n" + "="*60)
print("KEY CLINICAL FINDING (alpha=1000)")
print("="*60)
rows = [r for r in all_cal if r['alpha']==1000]
if rows:
    ns_sens = np.mean([r['no_scaling'  ]['mel_sens'] for r in rows])
    ft_sens = np.mean([r['federated_T' ]['mel_sens'] for r in rows])
    ns_ece  = np.mean([r['no_scaling'  ]['ECE_test'] for r in rows])
    ft_ece  = np.mean([r['federated_T' ]['ECE_test'] for r in rows])
    print(f"  Without calibration : mel sensitivity = {ns_sens:.1%}, ECE = {ns_ece:.4f}")
    print(f"  Federated T scaling : mel sensitivity = {ft_sens:.1%}, ECE = {ft_ece:.4f}")
    print(f"  Improvement         : +{(ft_sens-ns_sens)*100:.1f}pp sensitivity, "
          f"-{(ns_ece-ft_ece)/ns_ece*100:.1f}% ECE")
    print()
    print("  Paper sentence:")
    print(f"  'Federated temperature scaling improves melanoma sensitivity")
    print(f"   from {ns_sens:.1%} to {ft_sens:.1%} at a 70% confidence threshold,")
    print(f"   while reducing ECE from {ns_ece:.4f} to {ft_ece:.4f}.'")

print(f"\nAll results saved to: {RESULTS_DIR}")
print("NEXT: Upload output as 'feduca-phase5-results'")
print("Then ask for figure generation (all 8 figures from Phases 2-5).")